## 1. Ước lượng Bình phương tối thiểu (OLS)

Mục tiêu của phương pháp OLS là tìm vector tham số $\hat{\beta}$ sao cho tối thiểu hóa tổng bình phương các sai số (SSR). Vector $\hat{\beta}$ được tính toán dựa trên hệ phương trình chuẩn (Normal Equations):

**Công thức toán học:**
$$\hat{\beta} = (X^T X)^{-1} X^T y$$

**Các thành phần chính:**
- $X$: Ma trận thiết kế (Design matrix).
- $y$: Vector giá trị quan sát thực tế.
- $\hat{\beta}$: Vector các hệ số ước lượng (intercept và slopes).

## 2. Ma trận Hat (Projection Matrix)

Ma trận Hat ($H$) đóng vai trò ánh xạ vector giá trị quan sát $y$ tới vector giá trị dự báo $\hat{y}$. Về mặt hình học, nó thực hiện phép chiếu trực giao $y$ lên không gian cột của ma trận $X$.

**Công thức toán học:**
$$H = X(X^T X)^{-1} X^T$$
$$\hat{y} = H y$$

**Các tính chất quan trọng:**
1. **Tính đối xứng (Symmetry):** $H = H^T$.
2. **Tính lũy đẳng (Idempotency):** $H^2 = H$ (Việc thực hiện phép chiếu nhiều lần lên cùng một không gian sẽ cho kết quả không đổi).

*Lưu ý: Trong phần cài đặt, chúng mình sử dụng giả nghịch đảo `np.linalg.pinv` để đảm bảo tính ổn định số học khi ma trận $X^T X$ bị suy biến hoặc có đa cộng tuyến.*

In [3]:
import sys
import os
import numpy as np
import importlib

# Ensure path to import .py from part1 / Đảm bảo đường dẫn chính xác
if os.path.abspath("..") not in sys.path:
    sys.path.append(os.path.abspath(".."))

# Reload modules / Nạp lại các hàm mới nhất
import part1.ols_implementation

importlib.reload(part1.ols_implementation)
from part1.ols_implementation import ols_fit, hat_matrix

# 1. Demo OLS Fit
X_demo = np.array([[1, 1], [1, 2], [1, 3], [1, 4]])
y_demo = np.array([5.1, 8.2, 10.9, 14.3])

beta_hat, sigma2 = ols_fit(X_demo, y_demo)
print("--- OLS Result ---")
print(f"Beta coefficients: {beta_hat}")

# 2. Demo Hat Matrix
H = hat_matrix(X_demo)
print("\n--- Hat Matrix Properties ---")
print(f"Matrix Shape: {H.shape}")
# Check idempotency: H*H should be equal to H
is_idempotent = np.allclose(H @ H, H)
print(f"Is Idempotent (H^2 = H): {is_idempotent}")

--- OLS Result ---
Beta coefficients: [2.05 3.03]

--- Hat Matrix Properties ---
Matrix Shape: (4, 4)
Is Idempotent (H^2 = H): True


## 3. Đánh giá Mô hình (Model Evaluation Metrics)

Sau khi tìm được các hệ số ước lượng, chúng ta cần đánh giá xem mô hình fit với dữ liệu tốt đến mức nào bằng các độ đo thống kê. Trong hàm `model_metrics(y, y_hat, p)`, $p$ là số lượng biến độc lập (không bao gồm intercept).

**Các công thức tính toán:**
- **RSS (Residual Sum of Squares):** $RSS = \sum(y_i - \hat{y}_i)^2$
- **TSS (Total Sum of Squares):** $TSS = \sum(y_i - \bar{y})^2$
- **$R^2$ (Hệ số xác định):** $R^2 = 1 - \frac{RSS}{TSS}$
- **Adjusted $R^2$ (Hệ số $R^2$ hiệu chỉnh):** Phạt mô hình nếu thêm biến không hiệu quả. 
  $$\text{Adjusted } R^2 = 1 - \frac{n - 1}{n - p - 1}(1 - R^2)$$
- **F-statistic:** Kiểm định mức độ ý nghĩa tổng thể của mô hình.
  $$F = \frac{(TSS - RSS) / p}{RSS / (n - p - 1)}$$

In [4]:
from part1.ols_implementation import model_metrics

# 1. Tạo y_hat từ kết quả OLS ở phần trên
# Giả sử X_demo và beta_hat đã được tính ở phần 1
y_hat_demo = X_demo @ beta_hat

# p là số lượng biến độc lập (không tính cột intercept là cột toàn số 1)
p_demo = X_demo.shape[1] - 1

# 2. Tính toán Metrics
metrics = model_metrics(y_demo, y_hat_demo, p_demo)

# 3. In kết quả
print("\n--- Model Evaluation Metrics ---")
for metric_name, value in metrics.items():
    print(f"{metric_name}: {value:.4f}")


--- Model Evaluation Metrics ---
RSS: 0.0830
TSS: 45.9875
R2: 0.9982
Adjusted_R2: 0.9973
F_statistic: 1106.1325


## 4. Suy diễn thống kê cho các hệ số (Statistical Inference)

Để xác định xem một biến độc lập có thực sự ảnh hưởng đến biến phụ thuộc hay không, chúng ta thực hiện kiểm định giả thuyết thống kê cho từng hệ số $\beta_j$.

**Các bước tính toán:**
1. **Ma trận hiệp phương sai:** $Var(\hat{\beta}_{OLS} | X) = \sigma^2 (X^T X)^{-1}$
2. **Sai số chuẩn (Standard Error - SE):** Lấy căn bậc hai các phần tử trên đường chéo chính của ma trận hiệp phương sai. $SE(\hat{\beta}_j) = \sqrt{Var(\hat{\beta}_j)}$
3. **Giá trị t-statistic:** $t_j = \frac{\hat{\beta}_j}{SE(\hat{\beta}_j)}$
4. **Giá trị p-value:** Tra bảng phân phối Student-t với bậc tự do là $n - p - 1$. Nếu p-value < 0.05, hệ số đó có ý nghĩa thống kê ở mức độ tin cậy 95%.
5. **Khoảng tin cậy 95% (Confidence Interval):** Ước lượng vùng chứa giá trị thực của $\beta_j$.

In [1]:
from part1.ols_implementation import coef_inference

# Chạy thử hàm suy diễn thống kê với dữ liệu demo
inference_results = coef_inference(X_demo, y_demo, beta_hat, sigma2)

print("\n--- Coefficient Statistical Inference ---")
print(f"Standard Errors (SE): {inference_results['SE']}")
print(f"t-statistics: {inference_results['t_stats']}")
print(f"p-values: {inference_results['p_values']}")
print(f"95% CI Lower Bound: {inference_results['CI_lower']}")
print(f"95% CI Upper Bound: {inference_results['CI_upper']}")

ModuleNotFoundError: No module named 'part1'